# Electoral Gender Quotas and Democratic Legitimacy

**Authors:** Amanda Clayton, Diana Z. O'Brien, Jennifer M. Piscopo

**Journal:** American Political Science Review (2026)

This notebook reproduces the analysis from the paper above.
It was auto-generated by [PoliRep](https://polirep.org).

---

## Setup

Install packages and import standard libraries.

In [ ]:
!pip install -q pandas numpy matplotlib scipy statsmodels pyfixest

import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

## Configuration & Constants

Paper-specific constants used by the analysis code below.

In [ ]:
# Paper-specific constants (extracted from config.py)

# Colab-friendly data path (replaces local PAPER_ROOT-based path)
DATA_CONVERTED = Path("data/converted")
OUTPUT_DIR = Path("output")
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"

TREATMENTS = {
    1: "All-Male Panel (Sexual Harassment)",
    2: "Gender-Balanced Panel (Sexual Harassment)",
    3: "Quota-Gender-Balanced Panel (Sexual Harassment)",
    4: "All-Male Panel (Animal Mistreatment)",
    5: "Gender-Balanced Panel (Animal Mistreatment)",
    6: "Quota-Gender-Balanced Panel (Animal Mistreatment)",
}
HARASSMENT_TREATMENTS = [1, 2, 3]
ANIMAL_TREATMENTS = [4, 5, 6]
SUBSTANTIVE_LEG_VARS_HARASSMENT = ["decide_citizen", "decide_women", "fair_women"]
SUBSTANTIVE_LEG_VARS_ANIMAL = ["decide_citizen", "decide_animal", "fair_animal"]
PROCEDURAL_LEG_VARS = ["decisionprocess", "council_trust", "overturnR"]
SPANISH_FAIRNESS_FEMININE = {
    "Muy injusta": 1,
    "Algo injusta": 2,
    "Algo justa": 3,
    "Muy justa": 4,
}
SPANISH_FAIRNESS_MASCULINE = {
    "Muy injusto": 1,
    "Algo  injusto": 2,
    "Algo injusto": 2,
    "Algo justo": 3,
    "Muy justo": 4,
}
SPANISH_AGREEMENT = {
    "Muy en desacuerdo": 1,
    "Algo en desacuerdo": 2,
    "Algo de acuerdo": 3,
    "Muy de acuerdo": 4,
}
PORTUGUESE_FAIRNESS_FEMININE = {
    "Muito injusta": 1,
    "Um pouco injusta": 2,
    "Um pouco justa": 3,
    "Um pouca justa": 3,
    "Muito justa": 4,
}
PORTUGUESE_FAIRNESS_MASCULINE = {
    "Muito injusto": 1,
    "Um pouco injusto": 2,
    "Um pouco justo": 3,
    "Muito justo": 4,
}
PORTUGUESE_AGREEMENT = {
    "Discordo totalmente": 1,
    "Discordo": 2,
    "Concordo": 3,
    "Concordo totalmente": 4,
}
ENGLISH_FAIRNESS = {
    "Very unfair": 1,
    "Unfair": 2,
    "Somewhat fair": 3,
    "Fair": 3,
    "Very fair": 4,
}
ENGLISH_AGREEMENT = {
    "Strongly disagree": 1,
    "Disagree": 2,
    "Agree": 3,
    "Strongly agree": 4,
    "Strongly Agree": 4,
}
GENDER_MAPPINGS = {
    "Mujer": 1,
    "Woman": 1,
    "Female": 1,
    "Feminino": 1,
    "Hombre": 0,
    "Man": 0,
    "Male": 0,
    "Masculino": 0,
    "Other": None,
    "Otro/ No binario": None,
}
IDEOLOGY_MAPPINGS = {
    "Left\n(0)": 0,
    "Right\n(10)": 10,
}
COUNTRY_CONFIGS = {
    "Argentina": {
        "treat_var": "treat",
        "fairness_map": SPANISH_FAIRNESS_FEMININE | SPANISH_FAIRNESS_MASCULINE,
        "agreement_map": SPANISH_AGREEMENT,
        "manip_check_harass": "Requerir capacitación en acoso sexual.",
        "manip_check_animal": "Requerir capacitación en maltrato animal.",
        "duration_filter": None,
        "gender_var": "gender",
        "ideology_var": "leftright_1",
    },
    "Brazil": {
        "treat_var": "treat",
        "fairness_map": PORTUGUESE_FAIRNESS_FEMININE | PORTUGUESE_FAIRNESS_MASCULINE,
        "agreement_map": PORTUGUESE_AGREEMENT,
        "manip_check_harass": "Exigir treinamento sobre assédio sexual.",
        "manip_check_animal": "Exigir treinamento sobre maus-tratos a animais.",
        "duration_filter": None,
        "gender_var": "gender",
        "ideology_var": "leftright_1",
    },
    "Mexico": {
        "treat_var": "treat",
        "fairness_map": SPANISH_FAIRNESS_FEMININE | SPANISH_FAIRNESS_MASCULINE,
        "agreement_map": SPANISH_AGREEMENT,
        "manip_check_harass": None,  # Applied implicitly through subsetting
        "manip_check_animal": None,
        "duration_filter": 350,
        "gender_var": "gender",
        "ideology_var": "leftright_1",
    },
    "Peru": {
        "treat_var": "treat",
        "fairness_map": SPANISH_FAIRNESS_FEMININE | SPANISH_FAIRNESS_MASCULINE,
        "agreement_map": SPANISH_AGREEMENT,
        "manip_check_harass": None,
        "manip_check_animal": None,
        "duration_filter": 600,
        "gender_var": "gender",
        "ideology_var": "leftright_1",
    },
    "NZ": {
        "treat_var": "treat",
        "fairness_map": ENGLISH_FAIRNESS,
        "agreement_map": ENGLISH_AGREEMENT,
        "manip_check_harass": "Require sexual harassment training",
        "manip_check_animal": "Require animal mistreatment training",
        "duration_filter": None,
        "gender_var": "Q3.2",
        "ideology_var": "Q27.1",
    },
    "USA": {
        "treat_var": "treat",
        "fairness_map": ENGLISH_FAIRNESS,
        "agreement_map": ENGLISH_AGREEMENT,
        "manip_check_harass": "Require sexual harassment training",
        "manip_check_animal": "Require animal mistreatment training",
        "duration_filter": None,
        "gender_var": "gender",
        "ideology_var": "leftright_1",
    },
    "UK": {
        "treat_var": "Vignettes_DO",
        "fairness_map": ENGLISH_FAIRNESS,
        "agreement_map": ENGLISH_AGREEMENT,
        "manip_check_harass": None,
        "manip_check_animal": None,
        "duration_filter": 300,
        "gender_var": "gender",
        "ideology_var": "leftright_1",
    },
    "Spain": {
        "treat_var": "Vignettes_DO",
        "fairness_map": SPANISH_FAIRNESS_FEMININE | SPANISH_FAIRNESS_MASCULINE,
        "agreement_map": SPANISH_AGREEMENT,
        "manip_check_harass": None,
        "manip_check_animal": None,
        "duration_filter": 300,
        "gender_var": "gender",
        "ideology_var": "leftright_1",
    },
    "Portugal": {
        "treat_var": "Vignettes_DO",
        "fairness_map": PORTUGUESE_FAIRNESS_FEMININE | PORTUGUESE_FAIRNESS_MASCULINE,
        "agreement_map": PORTUGUESE_AGREEMENT,
        "manip_check_harass": None,
        "manip_check_animal": None,
        "duration_filter": 300,
        "gender_var": "gender",
        "ideology_var": "leftright_1",
    },
}

## Data Preparation


### Load Country-Level Aggregated Data

Load all 10 country-level aggregated datasets (ArgAgg, BrazilAgg,
MexAgg, PeruAgg, NZAgg, USAgg, UKAgg, SpainAgg, PortAgg, NorAusFranceAgg).
Each file contains pre-computed legitimacy indices and cleaned demographic
variables.

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load the merged analysis dataset from all country aggregated files.

    Returns:
        DataFrame with all countries merged together.
    """
    # Load all aggregated country datasets
    countries = [
        "Mexico",
        "Peru",
        "UK",
        "Spain",
        "Portugal",
        "USA",
        "Brazil",
        "Argentina",
        "NZ",
        "NorAusFrance",
    ]

    dfs = []
    for country in countries:
        df = load_country_data(country, use_agg=True)

        # Rename Vignettes_DO to treat for UK, Spain, Portugal
        if "Vignettes_DO" in df.columns and "treat" not in df.columns:
            df = df.rename(columns={"Vignettes_DO": "treat"})

        # Rename QCOUNTRY to Country for NorAusFrance
        if "QCOUNTRY" in df.columns and "Country" not in df.columns:
            df = df.rename(columns={"QCOUNTRY": "Country"})

        dfs.append(df)

    # Merge all datasets
    merged = pd.concat(dfs, ignore_index=True)

    return merged

In [ ]:
df = load_main_data()
print(f"Shape: {df.shape}")
print(f"Countries:")
print(df['Country'].value_counts())
print(f"\nTreatment distribution (raw, mixed types):")
print(df['treat'].value_counts())
display(df.head())


### Recode Treatment Variable

Convert text treatment labels to numeric codes. UK/Spain/Portugal use
"treat1"..."treat6" labels; other countries use numeric 1-6. Standardize
to numeric 1-6 for all countries.

In [ ]:
def prepare_analysis_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

    Args:
        df: Raw loaded DataFrame from load_main_data().

    Returns:
        Analysis-ready DataFrame with all filters and transformations applied.
    """
    df = df.copy()

    # Recode treatment variable (convert text to numeric if needed)
    treat_mapping = {
        "treat1": 1,
        "treat2": 2,
        "treat3": 3,
        "treat4": 4,
        "treat5": 5,
        "treat6": 6,
    }
    df["treat"] = df["treat"].replace(treat_mapping)
    df["treat"] = pd.to_numeric(df["treat"], errors="coerce")

    # Recode gender
    df["gender"] = df["gender"].replace(GENDER_MAPPINGS)
    df["gender"] = pd.to_numeric(df["gender"], errors="coerce")

    # Recode ideology
    df["leftright_1"] = df["leftright_1"].replace(IDEOLOGY_MAPPINGS)
    df["leftright_1"] = pd.to_numeric(df["leftright_1"], errors="coerce")

    return df

In [ ]:
sample = prepare_analysis_sample(load_main_data())
print(f"Treatment coding (after recode):")
print(sample['treat'].value_counts().sort_index())
print(f"\nTreatment type: {sample['treat'].dtype}")
print(f"Missing treatments: {sample['treat'].isna().sum()}")


### Recode Gender and Ideology

Recode gender from language-specific text labels to binary (1=Female, 0=Male).
Recode ideology from text labels to numeric 0-10 scale. Apply country-specific
mappings from config.

In [ ]:
def prepare_analysis_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

    Args:
        df: Raw loaded DataFrame from load_main_data().

    Returns:
        Analysis-ready DataFrame with all filters and transformations applied.
    """
    df = df.copy()

    # Recode treatment variable (convert text to numeric if needed)
    treat_mapping = {
        "treat1": 1,
        "treat2": 2,
        "treat3": 3,
        "treat4": 4,
        "treat5": 5,
        "treat6": 6,
    }
    df["treat"] = df["treat"].replace(treat_mapping)
    df["treat"] = pd.to_numeric(df["treat"], errors="coerce")

    # Recode gender
    df["gender"] = df["gender"].replace(GENDER_MAPPINGS)
    df["gender"] = pd.to_numeric(df["gender"], errors="coerce")

    # Recode ideology
    df["leftright_1"] = df["leftright_1"].replace(IDEOLOGY_MAPPINGS)
    df["leftright_1"] = pd.to_numeric(df["leftright_1"], errors="coerce")

    return df

In [ ]:
sample = prepare_analysis_sample(load_main_data())
print(f"Gender distribution:")
print(sample['gender'].value_counts())
print(f"\nIdeology summary:")
print(sample['leftright_1'].describe())
print(f"\nIdeology missing: {sample['leftright_1'].isna().sum()} ({sample['leftright_1'].isna().mean():.1%})")


### Compute Sample Statistics

Calculate sample size, country distribution, treatment balance, and
legitimacy outcome completeness. Report missing data patterns.

In [ ]:
def get_sample_stats(sample: pd.DataFrame) -> dict:
    """Compute summary statistics for the analysis sample.

    Args:
        sample: Prepared analysis sample from prepare_analysis_sample().

    Returns:
        Dictionary with keys like n_obs, n_countries, treatment_distribution, etc.
    """
    stats = {
        "n_obs": len(sample),
        "n_countries": sample["Country"].nunique() if "Country" in sample.columns else None,
        "n_complete_subleg": sample["SubLegStand"].notna().sum()
        if "SubLegStand" in sample.columns
        else None,
        "n_complete_proleg": sample["ProLegStand"].notna().sum()
        if "ProLegStand" in sample.columns
        else None,
    }

    if "treat" in sample.columns:
        stats["treatment_distribution"] = sample["treat"].value_counts().to_dict()

    if "Country" in sample.columns:
        stats["country_distribution"] = sample["Country"].value_counts().to_dict()

    return stats

In [ ]:
sample = prepare_analysis_sample(load_main_data())
stats = get_sample_stats(sample)
print(f"Total observations: {stats['n_obs']}")
print(f"Countries: {stats['n_countries']}")
print(f"Complete SubLegStand: {stats['n_complete_subleg']}")
print(f"Complete ProLegStand: {stats['n_complete_proleg']}")
print(f"\nCountry distribution:")
for country, count in sorted(stats['country_distribution'].items()):
    print(f"  {country}: {count}")
print(f"\nTreatment distribution:")
for treat, count in sorted(stats['treatment_distribution'].items()):
    print(f"  Treatment {treat}: {count}")


### Create Treatment Subsets

Split the merged dataset into sexual harassment vignettes (treatments 1-3)
and animal mistreatment vignettes (treatments 4-6) for separate analysis.
Australia/Norway/France only have harassment vignettes.

In [ ]:
def prepare_analysis_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

    Args:
        df: Raw loaded DataFrame from load_main_data().

    Returns:
        Analysis-ready DataFrame with all filters and transformations applied.
    """
    df = df.copy()

    # Recode treatment variable (convert text to numeric if needed)
    treat_mapping = {
        "treat1": 1,
        "treat2": 2,
        "treat3": 3,
        "treat4": 4,
        "treat5": 5,
        "treat6": 6,
    }
    df["treat"] = df["treat"].replace(treat_mapping)
    df["treat"] = pd.to_numeric(df["treat"], errors="coerce")

    # Recode gender
    df["gender"] = df["gender"].replace(GENDER_MAPPINGS)
    df["gender"] = pd.to_numeric(df["gender"], errors="coerce")

    # Recode ideology
    df["leftright_1"] = df["leftright_1"].replace(IDEOLOGY_MAPPINGS)
    df["leftright_1"] = pd.to_numeric(df["leftright_1"], errors="coerce")

    return df

In [ ]:
from src.config import HARASSMENT_TREATMENTS, ANIMAL_TREATMENTS
sample = prepare_analysis_sample(load_main_data())

harassment = sample[sample['treat'].isin(HARASSMENT_TREATMENTS)]
animal = sample[sample['treat'].isin(ANIMAL_TREATMENTS)]

print(f"Sexual harassment sample: {len(harassment)} obs")
print(f"  Treatment 1 (All-Male): {(harassment['treat']==1).sum()}")
print(f"  Treatment 2 (Gender-Balanced): {(harassment['treat']==2).sum()}")
print(f"  Treatment 3 (Quota): {(harassment['treat']==3).sum()}")

print(f"\nAnimal mistreatment sample: {len(animal)} obs")
print(f"  Treatment 4 (All-Male): {(animal['treat']==4).sum()}")
print(f"  Treatment 5 (Gender-Balanced): {(animal['treat']==5).sum()}")
print(f"  Treatment 6 (Quota): {(animal['treat']==6).sum()}")

print(f"\nCountries in harassment sample: {harassment['Country'].nunique()}")
print(f"Countries in animal sample: {animal['Country'].nunique()}")


## 1. Country-Level Data Processing


### 1. Country-Level Data Processing

Load and process 10 country datasets:
- Load raw survey data for each country
- Create factor-analyzed legitimacy indices (omega method)
- Standardize outcomes within country (mean=0, SD=1)
- Filter manipulation check failures and speedsters
- Output aggregated country files

Countries: Argentina, Australia, Brazil, France, Mexico, New Zealand,
Norway, Peru, Portugal, Spain, UK, USA

In [ ]:
def run():
    """Run country processing for all countries and save aggregated datasets."""
    print("Processing country datasets...")

    # Process standard countries
    standard_countries = ["Argentina", "Brazil", "Mexico", "Peru", "NZ", "USA"]
    for country in standard_countries:
        try:
            agg_df = process_standard_country(country)
            # Save aggregated dataset
            output_file = DATA_CONVERTED / f"{country.replace(' ', '')}Agg_python.csv"
            agg_df.to_csv(output_file, index=False)
            print(f"    Saved {output_file.name} ({len(agg_df)} rows)")
        except Exception as e:
            print(f"    ERROR processing {country}: {e}")

    # Process Vignettes_DO countries
    vignettes_countries = ["UK", "Spain", "Portugal"]
    for country in vignettes_countries:
        try:
            agg_df = process_vignettes_do_country(country)
            output_file = DATA_CONVERTED / f"{country}Agg_python.csv"
            agg_df.to_csv(output_file, index=False)
            print(f"    Saved {output_file.name} ({len(agg_df)} rows)")
        except Exception as e:
            print(f"    ERROR processing {country}: {e}")

    print("Country processing complete!")

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## 2. Merged Cross-Country Analysis


### 2. Merged Cross-Country Analysis

Combine country datasets and generate main figures:

**Figure 2:** Main treatment effects (sexual harassment vignettes)
- Substantive legitimacy by council composition
- Procedural legitimacy by council composition
- Pooled 12-country sample with 95% confidence intervals

**Figure 4:** USA vs UK comparison
- Treatment effects separately for two non-quota countries
- Highlights quota penalty in countries without statutory quotas

Treatment conditions:
1. All-male council (8 men)
2. Gender-balanced council (4 men, 4 women)
3. Quota-elected gender-balanced council (4 men, 4 women via quota rule)

In [ ]:
def run():
    """Run merged analysis and generate figures."""
    print("Running merged analysis...")

    # Load merged data
    merged = load_main_data()
    print(f"  Loaded {len(merged)} observations from {merged['Country'].nunique()} countries")

    # Prepare analysis sample
    merged = prepare_analysis_sample(merged)

    # Create figures
    print("  Creating Figure 2...")
    create_figure_2(merged)

    print("  Creating Figure 4...")
    create_figure_4(merged)

    print("Merged analysis complete!")

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## 3. Balance Tests (Randomization Check)


### 3. Balance Tests (Randomization Check)

Verify experimental randomization by comparing treatment groups on
pre-treatment covariates.

**Table SI.4:** Sexual harassment treatments (treats 1-3)
- Gender distribution across treatments
- Ideology (left-right scale) across treatments
- Country distribution across treatments

**Table SI.5:** Animal mistreatment treatments (treats 4-6)
- Same balance checks for non-gendered issue vignettes

Expected result: No significant differences (randomization successful)

## 4. Main Treatment Effects (T-Tests)


### 4. Main Treatment Effects (T-Tests)

Bivariate comparisons of legitimacy outcomes across treatment conditions.

**Table SI.6:** Sexual harassment vignettes (12 countries pooled)
- All-Male vs Gender-Balanced (treats 1 vs 2)
- All-Male vs Quota Gender-Balanced (treats 1 vs 3)
- Gender-Balanced vs Quota Gender-Balanced (treats 2 vs 3) = "quota penalty"
- For both substantive and procedural legitimacy

**Table SI.8:** USA and UK specific effects
- Country-specific treatment effects for two non-quota countries

**Table SI.9:** Animal mistreatment vignettes (9 countries)
- Same treatment comparisons for non-gendered issue

Method: Two-sample t-tests with 95% confidence intervals

## 5. OLS Regressions with Covariates


### 5. OLS Regressions with Covariates

Model-based estimates of treatment effects controlling for covariates
and country fixed effects.

**Table SI.10:** Substantive legitimacy regressions
- Model 1: All-Male vs Gender-Balanced + controls
- Model 2: All-Male vs Quota Gender-Balanced + controls
- Model 3: Gender-Balanced vs Quota Gender-Balanced + controls

**Table SI.11:** Procedural legitimacy regressions
- Same models as SI.10 but for procedural outcome

Controls:
- Gender (respondent)
- Political ideology (leftright_1, 0-10 scale)
- Country fixed effects (11 dummies)

Formula: Outcome ~ treat + gender + leftright_1 + Country

## 6. Robustness Checks (Excluding Political Left)


### 6. Robustness Checks (Excluding Political Left)

Test whether results hold when excluding liberal respondents
(ideology ≤ 4 on 0-10 scale).

**Table SI.12:** Substantive legitimacy (moderates/conservatives only)
- Same three models as SI.10
- Subset: leftright_1 > 4
- Expected: Effects attenuated but still significant

**Table SI.13:** Procedural legitimacy (moderates/conservatives only)
- Same three models as SI.11
- Subset: leftright_1 > 4

Purpose: Address concern that quota support driven by left-leaning
respondents only.